# Layer 2: per-farm spread and bundle selection under leave-one-farm-out

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

In [ ]:
PROJECT_ROOT = Path("/workspace")
CROSSFARM_DIR = PROJECT_ROOT / "output" / "14n_layer2_crossfarm_nested"
OUT_DIR = PROJECT_ROOT / "output" / "14n_layer2_crossfarm_nested_ablation"
LOG_DIR = PROJECT_ROOT / "output" / "logs"

OUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

RUN_TS = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
LOG_PATH = LOG_DIR / f"13b_layer2_actual_farm_holdout_ablation_{RUN_TAG}.log"

def log(msg: str):
    stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    line = f"[{stamp}] {msg}"
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")

def read_csv_auto(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix == ".gz":
        return pd.read_csv(path, compression="gzip")
    return pd.read_csv(path)

def read_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def safe_json_dump(obj, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def target_macro_f1(y_true, y_pred):
    return float(f1_score(y_true, y_pred, average="macro", zero_division=0))

log("Notebook started.")

In [ ]:
perfarm_path = CROSSFARM_DIR / "crossfarm_bestbundle_perfarm_metrics.csv"
preds_path = CROSSFARM_DIR / "crossfarm_fold_predictions.csv.gz"
metrics_path = CROSSFARM_DIR / "crossfarm_metrics_summary.csv"
manifest_path = CROSSFARM_DIR / "crossfarm_manifest.json"

perfarm_metrics_df = read_csv_auto(perfarm_path)
preds_df = read_csv_auto(preds_path)
metrics_df = read_csv_auto(metrics_path)
manifest = read_json(manifest_path)

log(f"Loaded per-farm metrics: {perfarm_metrics_df.shape}")
log(f"Loaded predictions: {preds_df.shape}")
display(perfarm_metrics_df)

In [ ]:
required_pred_cols = ["target_name", "heldout_farm", "model_name", "y_true", "y_pred"]
missing = [c for c in required_pred_cols if c not in preds_df.columns]
if missing:
    raise ValueError(f"Missing required prediction columns: {missing}")

preds_lr = preds_df[preds_df["model_name"] == "logreg_balanced"].copy()
heldout_farms = sorted(preds_lr["heldout_farm"].astype(str).dropna().unique().tolist())
log(f"Heldout farms present: {heldout_farms}")

for target_name, grp in preds_lr.groupby("target_name"):
    pooled = target_macro_f1(grp["y_true"].to_numpy(), grp["y_pred"].to_numpy())
    ref = metrics_df[(metrics_df["target_name"] == target_name) &
                     (metrics_df["model_name"] == "logreg_balanced")]["macro_f1"]
    ref = float(ref.iloc[0]) if len(ref) else float("nan")
    log(f"{target_name}: pooled macro-F1 recomputed={pooled:.4f} vs summary={ref:.4f}")

In [ ]:
floor_rows = []
for (target_name, heldout_farm), grp in preds_lr.groupby(["target_name", "heldout_farm"]):
    y_true = grp["y_true"].to_numpy()
    own = np.full_like(y_true, pd.Series(y_true).mode().iloc[0])
    floor_rows.append({
        "target_name": target_name, "heldout_farm": heldout_farm, "n_samples": int(len(grp)),
        "macro_f1": target_macro_f1(y_true, grp["y_pred"].to_numpy()),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, grp["y_pred"].to_numpy())),
        "training_majority_floor_macro_f1": target_macro_f1(y_true, grp["y_dummy"].to_numpy()),
        "own_majority_floor_macro_f1": target_macro_f1(y_true, own),
        "n_positive": int(y_true.sum()) if grp["task_type"].iloc[0] == "binary" else None,
    })
perfarm_floor_df = pd.DataFrame(floor_rows)
display(perfarm_floor_df.round(3))

In [ ]:
selection_df = read_csv_auto(CROSSFARM_DIR / "crossfarm_bundle_selection_scores.csv")
selection_df["vocabulary"] = np.where(selection_df["bundle_name"].str.contains("high_bias_filtered"), "filtered", "full")
selection_df["context"] = selection_df["bundle_name"].str.split("__").str[1]

chosen = preds_lr.groupby(["heldout_farm", "target_name"])["bundle_name"].first().reset_index()
chosen["vocabulary"] = np.where(chosen["bundle_name"].str.contains("high_bias_filtered"), "filtered", "full")
chosen["context"] = chosen["bundle_name"].str.split("__").str[1]
display(chosen)
display(chosen["vocabulary"].value_counts())
display(chosen["context"].value_counts())

vocab_scores = (selection_df.groupby(["target_name", "heldout_farm", "vocabulary"])["inner_macro_f1"].max()
                .unstack().groupby("target_name").mean())
vocab_scores["filtered_minus_full"] = vocab_scores["filtered"] - vocab_scores["full"]
context_scores = (selection_df.groupby(["target_name", "heldout_farm", "context"])["inner_macro_f1"].max()
                  .unstack().groupby("target_name").mean())
display(vocab_scores.round(3))
display(context_scores.round(3))

In [ ]:
spread_rows = []
for target_name, grp in perfarm_metrics_df.groupby("target_name"):
    spread_rows.append({
        "target_name": target_name,
        "n_farms": int(grp["heldout_farm"].nunique()),
        "macro_f1_mean": float(grp["macro_f1"].mean()),
        "macro_f1_std": float(grp["macro_f1"].std(ddof=0)),
        "macro_f1_min": float(grp["macro_f1"].min()),
        "macro_f1_max": float(grp["macro_f1"].max()),
        "macro_f1_range": float(grp["macro_f1"].max() - grp["macro_f1"].min()),
        "balanced_accuracy_mean": float(grp["balanced_accuracy"].mean()),
        "balanced_accuracy_std": float(grp["balanced_accuracy"].std(ddof=0)),
        "balanced_accuracy_min": float(grp["balanced_accuracy"].min()),
        "balanced_accuracy_max": float(grp["balanced_accuracy"].max()),
        "balanced_accuracy_range": float(grp["balanced_accuracy"].max() - grp["balanced_accuracy"].min()),
    })
spread_df = pd.DataFrame(spread_rows).sort_values("target_name").reset_index(drop=True)
display(spread_df)

In [ ]:
def bootstrap_metric(grp, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    y_true = grp["y_true"].to_numpy()
    y_pred = grp["y_pred"].to_numpy()
    n = len(grp)
    acc_vals, ba_vals, f1_vals = [], [], []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt, yp = y_true[idx], y_pred[idx]
        acc_vals.append(accuracy_score(yt, yp))
        ba_vals.append(balanced_accuracy_score(yt, yp))
        f1_vals.append(target_macro_f1(yt, yp))
    def q(x):
        return float(np.quantile(x, 0.025)), float(np.quantile(x, 0.975))
    acc_lo, acc_hi = q(acc_vals)
    ba_lo, ba_hi = q(ba_vals)
    f1_lo, f1_hi = q(f1_vals)
    return {
        "accuracy_ci_low": acc_lo, "accuracy_ci_high": acc_hi,
        "balanced_accuracy_ci_low": ba_lo, "balanced_accuracy_ci_high": ba_hi,
        "macro_f1_ci_low": f1_lo, "macro_f1_ci_high": f1_hi,
    }

bootstrap_rows = []
for target_name, grp in preds_lr.groupby("target_name"):
    ci = bootstrap_metric(grp, n_boot=1000, seed=42)
    bootstrap_rows.append({"target_name": target_name, "heldout_farm": "POOLED", **ci})
    for heldout_farm in ["1", "2", "3"]:
        fg = grp[grp["heldout_farm"].astype(str) == heldout_farm]
        if fg.empty:
            continue
        ci = bootstrap_metric(fg, n_boot=1000, seed=42)
        bootstrap_rows.append({"target_name": target_name, "heldout_farm": heldout_farm, **ci})
bootstrap_ci_df = pd.DataFrame(bootstrap_rows).sort_values(["target_name", "heldout_farm"]).reset_index(drop=True)
display(bootstrap_ci_df)

In [ ]:
perfarm_out = OUT_DIR / "actual_farm_holdout_best_perfarm.csv"
spread_out = OUT_DIR / "actual_farm_holdout_spread_summary.csv"
bootstrap_out = OUT_DIR / "actual_farm_holdout_bootstrap_ci.csv"
manifest_out_path = OUT_DIR / "actual_farm_holdout_manifest.json"

perfarm_metrics_df.to_csv(perfarm_out, index=False)
spread_df.to_csv(spread_out, index=False)
bootstrap_ci_df.to_csv(bootstrap_out, index=False)
perfarm_floor_df.to_csv(OUT_DIR / "actual_farm_holdout_perfarm_floors.csv", index=False)
chosen.to_csv(OUT_DIR / "selected_bundles_by_fold.csv", index=False)
vocab_scores.reset_index().to_csv(OUT_DIR / "inner_scores_by_vocabulary.csv", index=False)
context_scores.reset_index().to_csv(OUT_DIR / "inner_scores_by_context.csv", index=False)

n_total = int(preds_lr.groupby("target_name").size().max())
manifest_out = {
    "build_ts_utc": RUN_TS,
    "source_crossfarm_dir": str(CROSSFARM_DIR),
    "heldout_farms_reported": ["1", "2", "3"],
    "unknown_farm_used_as_fold": False,
    "n_known_farm_clips_per_target": n_total,
    "metric": "macro_f1 (average='macro' for all targets)",
    "outputs": {
        "best_perfarm_csv": perfarm_out.name,
        "spread_summary_csv": spread_out.name,
        "bootstrap_ci_csv": bootstrap_out.name,
        "perfarm_floors_csv": "actual_farm_holdout_perfarm_floors.csv",
        "selected_bundles_csv": "selected_bundles_by_fold.csv",
        "inner_scores_by_vocabulary_csv": "inner_scores_by_vocabulary.csv",
        "inner_scores_by_context_csv": "inner_scores_by_context.csv",
    },
}
safe_json_dump(manifest_out, manifest_out_path)
log(f"Saved spread summary: {spread_out}")
log("Notebook completed.")